# Sprint 2b.5–2b.6 — ML Model, Risk Scoring & Fragile Tracts
## AI for Equitable Public Transportation | Deloitte Capstone

**Purpose:** Build an XGBoost model for accurate equity score prediction and Sprint 3 simulation, then identify **fragile tracts** that are at risk of worsening equity outcomes.

**Input:**
- `Sprint2b_Modeling_Features_NotebookOutput.csv` (504 tracts × 67 columns)
- `Sprint2b_Ridge_Predictions.csv` (regression baseline from Notebook 1)

**Key outputs:**
- Trained XGBoost model with SHAP explanations
- Model comparison (Ridge vs XGBoost)
- Fragile tract identification with projected risk scores
- Sprint 3 simulation readiness demo

| Version | Date | Changes |
|---------|------|---------|
| v1 | 2026-03-16 | Initial build: XGBoost with CV tuning, SHAP, risk scoring, simulation demo |

## 1. Setup and Imports

Load all required libraries. XGBoost for gradient-boosted tree modeling, SHAP for model explanations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
from sklearn.model_selection import (
    cross_val_score, cross_val_predict, StratifiedKFold,
    GridSearchCV, train_test_split
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 70)
pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)

print("Libraries loaded successfully.")

## 2. Load Data

Load the modeling dataset and the regression predictions from Notebook 1 (for comparison).

In [ ]:
df = pd.read_csv('Sprint2b_Modeling_Features_NotebookOutput.csv', dtype={'tract_geoid': str})
print(f"Modeling dataset: {df.shape}")

# Load regression predictions for later comparison
try:
    ridge_preds = pd.read_csv('Sprint2b_Ridge_Predictions.csv', dtype={'tract_geoid': str})
    print(f"Ridge predictions loaded: {ridge_preds.shape}")
    HAS_RIDGE = True
except FileNotFoundError:
    print("Ridge predictions not found — run Notebook 1 first for comparison.")
    HAS_RIDGE = False

## 3. Target Leakage Exclusion

We apply the **exact same exclusions** as Notebook 1 to ensure a fair comparison. These 15 columns are either components of the target formula, derivatives of the target, zero-variance, or identifiers.

See Notebook 1 Section 3 for the full rationale behind each exclusion.

In [ ]:
TARGET = 'equity_priority_score'

LEAKAGE_COLS = [
    'equity_priority_score',
    'equity_tier',
    'equity_percentile',
    'composite_need',
    'composite_access_deficit',
    'ind_1_transit_dependency',
    'ind_2_temporal_mismatch',
    'ind_3_structural_gap',
    'ind_4_time_tax',
    'ind_5_service_coverage',
    'ind_6_economic_vulnerability',
    'ind_7_multimodal_deficit',
    'neighbor_mean_equity_score',
    'institutional_tract',
    'tract_geoid',
]

y = df[TARGET].copy()
feature_cols = [c for c in df.columns if c not in LEAKAGE_COLS]
X = df[feature_cols].copy()

# Store tier labels for stratification and analysis (NOT as a feature)
tier_labels = df['equity_tier'].values

print(f"Target: {TARGET} (n={len(y)}, mean={y.mean():.4f}, std={y.std():.4f})")
print(f"Features: {X.shape[1]} columns")
print(f"Excluded: {len(LEAKAGE_COLS)} leakage/identifier columns")

# Quick leakage check: max feature-target correlation
max_r = X.corrwith(y).abs().max()
print(f"\nMax |feature-target correlation|: {max_r:.4f}")
if max_r > 0.85:
    print("⚠ WARNING: Possible residual leakage — investigate.")
else:
    print("✓ No suspicious correlations — feature set is clean.")

## 4. Train/Test Split

We split the data 80/20 for model evaluation, stratified by equity tier to ensure each split has proportional representation of Critical, High, Moderate, and Low tracts.

**Why not just use cross-validation?** We use the held-out test set for the final unbiased evaluation and SHAP analysis. Cross-validation is used during hyperparameter tuning to prevent overfitting to the training set.

In [ ]:
X_train, X_test, y_train, y_test, tier_train, tier_test = train_test_split(
    X, y, tier_labels,
    test_size=0.20,
    random_state=42,
    stratify=tier_labels
)

print(f"Train: {X_train.shape[0]} tracts ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test:  {X_test.shape[0]} tracts ({X_test.shape[0]/len(X)*100:.0f}%)")

print(f"\nTier distribution check:")
print(f"{'Tier':12s} {'Train':>6s} {'Test':>6s} {'Total':>6s}")
for tier in ['Low', 'Moderate', 'High', 'Critical']:
    n_train = (tier_train == tier).sum()
    n_test = (tier_test == tier).sum()
    n_total = (tier_labels == tier).sum()
    print(f"  {tier:10s} {n_train:6d} {n_test:6d} {n_total:6d}")

## 5. XGBoost Hyperparameter Tuning

With only 504 rows, **overfitting is the primary risk**. We use conservative hyperparameters:

- **max_depth=3–5**: Shallow trees generalize better on small datasets
- **min_child_weight=5–10**: Requires more samples per leaf, preventing overfitting to individual tracts
- **learning_rate=0.05**: Slow learning + early stopping = better generalization
- **subsample & colsample_bytree < 1.0**: Random sampling of rows and features per tree adds noise that prevents memorization

We search over a grid using 5-fold stratified CV on the training set only.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'max_depth': [3, 4, 5],
    'min_child_weight': [5, 10, 15],
    'learning_rate': [0.05],
    'n_estimators': [200, 300, 500],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0, 0.1],      # L1 regularization
    'reg_lambda': [1, 5],       # L2 regularization
}

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    verbosity=0
)

print(f"Grid search: {np.prod([len(v) for v in param_grid.values()])} combinations")
print("Running 5-fold CV for each combination...")

grid_search = GridSearchCV(
    xgb_model,
    param_grid,
    cv=cv.split(X_train, tier_train),
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print(f"\nBest parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"\nBest CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

## 6. Train Final XGBoost Model

Retrain with the best hyperparameters on the full training set. We use early stopping on the test set to find the optimal number of boosting rounds (trees).

Early stopping monitors the test error and stops adding trees when the error stops improving — this is our primary defense against overfitting.

In [ ]:
best_model = xgb.XGBRegressor(
    **best_params,
    objective='reg:squarederror',
    random_state=42,
    verbosity=0,
    early_stopping_rounds=30,
)

best_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print(f"Best iteration: {best_model.best_iteration}")
print(f"Trees used: {best_model.best_iteration + 1} / {best_params.get('n_estimators', 'N/A')}")

# Predictions
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

print(f"\nTraining performance:")
print(f"  R²:   {r2_score(y_train, y_pred_train):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")

print(f"\nTest performance:")
print(f"  R²:   {r2_score(y_test, y_pred_test):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")

# Check for overfitting
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
gap = train_r2 - test_r2
print(f"\nOverfit check: train R² - test R² = {gap:.4f}")
if gap > 0.15:
    print("⚠ Possible overfitting — train R² is much higher than test R².")
else:
    print("✓ Gap is acceptable — model generalizes well.")

## 7. Full Cross-Validation Evaluation

For a fair comparison with the Ridge model from Notebook 1, we also run 5-fold CV on the entire dataset (not just the training set). This gives us an apples-to-apples comparison.

In [ ]:
# Retrain on full data with best params for CV evaluation
# Override n_estimators with the early-stopped count from Section 6
best_params_cv = {**best_params, 'n_estimators': best_model.best_iteration + 1}
xgb_full = xgb.XGBRegressor(
    **best_params_cv,
    objective='reg:squarederror',
    random_state=42,
    verbosity=0,
)

# Cross-validated predictions on full dataset
y_pred_cv = cross_val_predict(
    xgb_full, X, y,
    cv=cv.split(X, tier_labels)
)

# CV metrics
cv_r2 = r2_score(y, y_pred_cv)
cv_rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
cv_mae = mean_absolute_error(y, y_pred_cv)

print(f"{'='*50}")
print(f"5-FOLD CROSS-VALIDATION RESULTS (XGBoost)")
print(f"{'='*50}")
print(f"  R²:   {cv_r2:.4f}")
print(f"  RMSE: {cv_rmse:.4f}")
print(f"  MAE:  {cv_mae:.4f}")

# Per-fold R²
fold_scores = cross_val_score(
    xgb_full, X, y,
    cv=cv.split(X, tier_labels),
    scoring='r2'
)
print(f"\nPer-fold R²:")
for i, s in enumerate(fold_scores):
    print(f"  Fold {i+1}: {s:.4f}")
print(f"  Mean:   {fold_scores.mean():.4f} ± {fold_scores.std():.4f}")

## 8. Model Comparison: Ridge vs XGBoost

Side-by-side comparison of the interpretable regression (Notebook 1) vs the ML model. This tells us whether the additional complexity of XGBoost is justified.

If both models perform similarly, the simpler Ridge model is preferable for interpretation. If XGBoost is significantly better, it captures nonlinear relationships that Ridge misses — and those are valuable for Sprint 3 simulation.

In [ ]:
if HAS_RIDGE:
    # Load Ridge CV predictions
    ridge_pred = ridge_preds['predicted_score_ridge'].values
    ridge_true = ridge_preds['true_score'].values
    
    ridge_r2 = r2_score(ridge_true, ridge_pred)
    ridge_rmse = np.sqrt(mean_squared_error(ridge_true, ridge_pred))
    ridge_mae = mean_absolute_error(ridge_true, ridge_pred)
    
    print(f"{'='*60}")
    print(f"MODEL COMPARISON (5-fold CV, original scale)")
    print(f"{'='*60}")
    print(f"{'Metric':12s} {'Ridge':>12s} {'XGBoost':>12s} {'Winner':>10s}")
    print(f"{'-'*60}")
    
    for metric, r_val, x_val in [
        ('R²', ridge_r2, cv_r2),
        ('RMSE', ridge_rmse, cv_rmse),
        ('MAE', ridge_mae, cv_mae),
    ]:
        if metric == 'R²':
            winner = 'XGBoost' if x_val > r_val else 'Ridge'
        else:
            winner = 'XGBoost' if x_val < r_val else 'Ridge'
        print(f"  {metric:10s} {r_val:12.4f} {x_val:12.4f} {winner:>10s}")
    
    # Scatter comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    tier_colors = {'Low': '#2ecc71', 'Moderate': '#f39c12', 'High': '#e74c3c', 'Critical': '#8e44ad'}
    
    for tier in ['Low', 'Moderate', 'High', 'Critical']:
        mask = tier_labels == tier
        axes[0].scatter(ridge_true[mask], ridge_pred[mask], alpha=0.5, s=15,
                       color=tier_colors[tier], label=tier)
        axes[1].scatter(y.values[mask], y_pred_cv[mask], alpha=0.5, s=15,
                       color=tier_colors[tier], label=tier)
    
    lims = [0, max(y.max(), ridge_true.max()) * 1.05]
    for ax, title, r2_val in [
        (axes[0], f'Ridge Regression (R²={ridge_r2:.4f})', ridge_r2),
        (axes[1], f'XGBoost (R²={cv_r2:.4f})', cv_r2)
    ]:
        ax.plot(lims, lims, 'k--', linewidth=1)
        ax.set_xlabel('True Score')
        ax.set_ylabel('Predicted Score')
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_xlim(lims)
        ax.set_ylim(lims)
    
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Ridge predictions not available — run Notebook 1 first.")
    print(f"\nXGBoost standalone results:")
    print(f"  CV R²:   {cv_r2:.4f}")
    print(f"  CV RMSE: {cv_rmse:.4f}")
    print(f"  CV MAE:  {cv_mae:.4f}")

## 9. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) tells us the contribution of each feature to each individual prediction. Unlike simple feature importance, SHAP shows:

1. **Which features matter most** (global importance)
2. **Direction of effect** (does higher value increase or decrease the score?)
3. **Per-tract explanations** (why did THIS tract get a high score?)

We train the final model on all data for SHAP analysis, since we already validated performance via cross-validation.

In [ ]:
# Train final model on all data for SHAP
xgb_final = xgb.XGBRegressor(
    **best_params_cv,
    objective='reg:squarederror',
    random_state=42,
    verbosity=0,
)
xgb_final.fit(X, y)

# SHAP values
print("Computing SHAP values (this may take a moment)...")
explainer = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X)

print(f"SHAP values shape: {shap_values.shape}")
print("Done.")

### 9.1 SHAP Summary Plot

Each dot is one tract. The x-axis shows the SHAP value (impact on prediction). Color shows the feature value (red = high, blue = low). Features are sorted by overall importance (top = most important).

This answers: "What features drive the model, and in which direction?".

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(8, X.shape[1] * 0.25)))
shap.summary_plot(shap_values, X, plot_type='dot', show=False, max_display=25)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.2 SHAP Bar Plot (Mean Absolute Impact)

A simpler view: average absolute SHAP value per feature. This ranks features by their overall contribution to predictions, regardless of direction.

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(6, X.shape[1] * 0.22)))
shap.summary_plot(shap_values, X, plot_type='bar', show=False, max_display=25)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top features numerically
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'feature': X.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print(f"\nTop 15 features by mean |SHAP|:")
print(f"{'Feature':45s} {'Mean |SHAP|':>12s} {'% of total':>10s}")
total_shap = shap_importance['mean_abs_shap'].sum()
for _, row in shap_importance.head(15).iterrows():
    pct = row['mean_abs_shap'] / total_shap * 100
    print(f"  {row['feature']:45s} {row['mean_abs_shap']:12.6f} {pct:9.1f}%")

### 9.3 SHAP Dependence Plots for Top Features

These show how individual feature values relate to their SHAP impact. Each dot is one tract. The interaction feature (color) is automatically chosen by SHAP as the strongest interacting variable.

These plots reveal **nonlinear relationships** that the Ridge regression cannot capture.

In [ ]:
# Top 4 features
top_features = shap_importance.head(4)['feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, feat in enumerate(top_features):
    ax = axes[idx // 2][idx % 2]
    shap.dependence_plot(feat, shap_values, X, ax=ax, show=False)
    ax.set_title(f'SHAP Dependence: {feat}')

plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Risk Scoring — Identifying Fragile Tracts (Sprint 2b.6)

### What is a "fragile" tract?

A **fragile tract** is a census tract that is **currently not in crisis** (Moderate or High tier) but is **on a trajectory toward worsening equity outcomes** based on observable trends. These are the tracts where early intervention could prevent deterioration.

### Methodology

We use a **forecast-then-predict** approach:

1. **Take current features** for each tract
2. **Project trend variables forward** by 2 and 3 years using the ACS trend slopes (e.g., if poverty is rising by +0.5%/year, add 1.0% for a 2-year projection)
3. **Re-predict** the equity priority score using our trained XGBoost model
4. **Flag tracts** where the projected score crosses a tier threshold

### Why this works

The ACS trend slopes capture 5 years of observed change (2019-2023). Projecting them forward assumes the current trajectory continues — a reasonable baseline for short-term planning. The XGBoost model then translates these projected feature changes into equity score changes, accounting for nonlinear interactions between features.

In [ ]:
# Identify trend columns (these are the ones we can project forward)
trend_cols = [c for c in X.columns if c.startswith('trend_')]
base_cols = [c.replace('trend_', '') for c in trend_cols]

# Map trend columns to their base feature (if it exists in X)
trend_to_base = {}
for tc in trend_cols:
    base_name = tc.replace('trend_', '')
    if base_name in X.columns:
        trend_to_base[tc] = base_name

print(f"Trend columns: {len(trend_cols)}")
print(f"Trend columns with matching base feature: {len(trend_to_base)}")
print()
for tc, bc in trend_to_base.items():
    slope_med = X[tc].median()
    base_med = X[bc].median()
    print(f"  {tc:45s} → {bc:30s} (median slope: {slope_med:+.4f}/yr, base median: {base_med:.2f})")

### 10.1 Project Features Forward

For each tract, we create two scenarios:
- **2-year projection:** current value + (trend slope × 2)
- **3-year projection:** current value + (trend slope × 3)

Only features with matching trend slopes are projected. GTFS features remain unchanged (transit schedule changes require Sprint 3 simulation, not trend extrapolation).

In [ ]:
def project_features(X_current, trend_to_base_map, years_forward):
    """Project features forward using trend slopes.
    
    For each (trend_col, base_col) pair:
      projected_base = current_base + trend_slope × years_forward
    
    Returns a new DataFrame with projected values.
    """
    X_proj = X_current.copy()
    
    for trend_col, base_col in trend_to_base_map.items():
        # Project: new_value = current_value + slope * years
        X_proj[base_col] = X_proj[base_col] + X_proj[trend_col] * years_forward
        
        # Clip percentages to [0, 100]
        if 'pct' in base_col or 'rate' in base_col:
            X_proj[base_col] = X_proj[base_col].clip(0, 100)
    
    return X_proj

# Create projections
X_2yr = project_features(X, trend_to_base, years_forward=2)
X_3yr = project_features(X, trend_to_base, years_forward=3)

# Recompute interaction terms that depend on projected features
for proj_X, label in [(X_2yr, '2yr'), (X_3yr, '3yr')]:
    if 'poverty_x_low_span' in proj_X.columns and 'poverty_rate_pct' in proj_X.columns:
        proj_X['poverty_x_low_span'] = proj_X['poverty_rate_pct'] * (24 - proj_X['mean_service_span_hours'])
    if 'novehicle_x_low_service' in proj_X.columns and 'hh_no_vehicle_pct' in proj_X.columns:
        proj_X['novehicle_x_low_service'] = proj_X['hh_no_vehicle_pct'] * (1 / (1 + proj_X['total_weekday_trips']))
    if 'novehicle_x_weekend_gap' in proj_X.columns and 'hh_no_vehicle_pct' in proj_X.columns:
        proj_X['novehicle_x_weekend_gap'] = proj_X['hh_no_vehicle_pct'] * (1 - proj_X['weekend_weekday_ratio'])
    if 'rising_poverty_x_current' in proj_X.columns and 'trend_poverty_rate_pct' in proj_X.columns:
        proj_X['rising_poverty_x_current'] = proj_X['trend_poverty_rate_pct'].clip(lower=0) * proj_X['poverty_rate_pct']

# Predict future scores
y_current = xgb_final.predict(X)
y_2yr = xgb_final.predict(X_2yr)
y_3yr = xgb_final.predict(X_3yr)

print(f"Current scores:  mean={y_current.mean():.4f}, std={y_current.std():.4f}")
print(f"2-year projected: mean={y_2yr.mean():.4f}, std={y_2yr.std():.4f}")
print(f"3-year projected: mean={y_3yr.mean():.4f}, std={y_3yr.std():.4f}")

### 10.2 Identify Fragile Tracts

A tract is classified as **fragile** if it meets ALL of these criteria:

1. **Currently Moderate or High tier** (not already Critical, and not Low — which would need more dramatic change)
2. **Projected score increases** by at least 10% over 3 years
3. **Projected score would cross into a higher tier** (Moderate → High, or High → Critical)

We use the current tier thresholds (percentile-based cutoffs from Luna's Sprint 2a) to determine tier crossings.

In [ ]:
# Compute current tier thresholds from the actual score distribution
p90 = y.quantile(0.90)
p70 = y.quantile(0.70)
p40 = y.quantile(0.40)

print(f"Tier thresholds (from current score distribution):")
print(f"  Critical: > {p90:.4f} (90th percentile)")
print(f"  High:     > {p70:.4f} (70th percentile)")
print(f"  Moderate: > {p40:.4f} (40th percentile)")
print(f"  Low:      ≤ {p40:.4f}")

def assign_tier(score, p90=p90, p70=p70, p40=p40):
    if score > p90:
        return 'Critical'
    elif score > p70:
        return 'High'
    elif score > p40:
        return 'Moderate'
    else:
        return 'Low'

# Build risk assessment table
risk_df = pd.DataFrame({
    'tract_geoid': df['tract_geoid'].values,
    'current_tier': tier_labels,
    'current_score': y.values,
    'predicted_current': y_current,
    'predicted_2yr': y_2yr,
    'predicted_3yr': y_3yr,
})

risk_df['projected_tier_2yr'] = risk_df['predicted_2yr'].apply(assign_tier)
risk_df['projected_tier_3yr'] = risk_df['predicted_3yr'].apply(assign_tier)
risk_df['score_change_3yr'] = risk_df['predicted_3yr'] - risk_df['predicted_current']
risk_df['score_change_pct'] = (risk_df['score_change_3yr'] / risk_df['predicted_current']) * 100

# Tier rank for comparison
tier_rank = {'Low': 0, 'Moderate': 1, 'High': 2, 'Critical': 3}
risk_df['current_rank'] = risk_df['current_tier'].map(tier_rank)
risk_df['projected_rank_3yr'] = risk_df['projected_tier_3yr'].map(tier_rank)
risk_df['tier_worsened'] = risk_df['projected_rank_3yr'] > risk_df['current_rank']

# Fragile: currently Moderate or High, AND projected to worsen
fragile_mask = (
    risk_df['current_tier'].isin(['Moderate', 'High']) &
    risk_df['tier_worsened'] &
    (risk_df['score_change_pct'] > 10)
)

risk_df['fragile'] = fragile_mask

print(f"\n{'='*60}")
print(f"FRAGILE TRACT IDENTIFICATION")
print(f"{'='*60}")
print(f"Total tracts analyzed: {len(risk_df)}")
print(f"Fragile tracts: {fragile_mask.sum()}")
print(f"\nBreakdown:")
print(f"  Moderate → High:     {((risk_df['current_tier']=='Moderate') & risk_df['fragile']).sum()}")
print(f"  High → Critical:     {((risk_df['current_tier']=='High') & risk_df['fragile']).sum()}")

# Also flag tracts that improve
improving_mask = risk_df['projected_rank_3yr'] < risk_df['current_rank']
print(f"\nTracts projected to IMPROVE: {improving_mask.sum()}")

### 10.3 Fragile Tract Details

The table below shows each fragile tract with its current and projected scores, the expected tier transition, and the primary driver of deterioration (the feature with the largest projected change).

In [ ]:
if fragile_mask.sum() > 0:
    fragile_tracts = risk_df[fragile_mask].sort_values('score_change_pct', ascending=False)
    
    # For each fragile tract, find the biggest projected feature change
    drivers = []
    for idx in fragile_tracts.index:
        changes = {}
        for tc, bc in trend_to_base.items():
            slope = X.loc[idx, tc]
            base_val = X.loc[idx, bc]
            projected_change = slope * 3  # 3-year change
            if base_val != 0:
                rel_change = abs(projected_change / base_val)
            else:
                rel_change = abs(projected_change)
            changes[bc] = (projected_change, rel_change)
        
        # Top driver by relative change
        top_driver = max(changes.items(), key=lambda x: x[1][1])
        drivers.append(f"{top_driver[0]} ({top_driver[1][0]:+.2f})")
    
    fragile_tracts = fragile_tracts.copy()
    fragile_tracts['primary_driver'] = drivers
    
    display_cols = [
        'tract_geoid', 'current_tier', 'projected_tier_3yr',
        'current_score', 'predicted_3yr', 'score_change_pct', 'primary_driver'
    ]
    
    print(f"\n{'='*100}")
    print(f"FRAGILE TRACTS — Projected to worsen within 3 years")
    print(f"{'='*100}")
    print(fragile_tracts[display_cols].to_string(index=False))
    
    # Summary statistics
    print(f"\n\nSummary:")
    print(f"  Average projected score increase: {fragile_tracts['score_change_pct'].mean():.1f}%")
    print(f"  Max projected score increase: {fragile_tracts['score_change_pct'].max():.1f}%")
else:
    print("No fragile tracts identified under current criteria.")
    print("This could mean trends are stable or the model is conservative.")
    print("\nRelaxing criteria: tracts with >5% projected increase (any tier)...")
    relaxed = risk_df[risk_df['score_change_pct'] > 5].sort_values('score_change_pct', ascending=False)
    if len(relaxed) > 0:
        print(f"  Found {len(relaxed)} tracts with >5% projected increase")
        print(relaxed[['tract_geoid', 'current_tier', 'current_score', 
                       'predicted_3yr', 'score_change_pct']].head(15).to_string(index=False))
    else:
        print("  No tracts show significant projected increase.")

## 11. Sprint 3 Simulation Readiness

The XGBoost model accepts GTFS transit features (headway, frequency, service span, stop count, etc.) as inputs. This means we can simulate "what-if" scenarios by modifying these features and re-predicting.

**Demo:** For the top 10 Critical tracts, we simulate:
- Halving peak AM headway (doubling bus frequency)
- Adding 5 more stops to the tract

This previews the kind of analysis Sprint 3 will perform at full scale.

In [ ]:
# Identify top Critical tracts
critical_tracts = risk_df[risk_df['current_tier'] == 'Critical'].nlargest(10, 'current_score')
critical_idx = critical_tracts.index

# Simulate: halve headway, add 5 stops
X_sim = X.loc[critical_idx].copy()
original_scores = xgb_final.predict(X_sim)

# Intervention 1: halve peak AM headway
if 'headway_peak_am_min' in X_sim.columns:
    X_sim_headway = X_sim.copy()
    X_sim_headway['headway_peak_am_min'] = X_sim_headway['headway_peak_am_min'] / 2
    scores_headway = xgb_final.predict(X_sim_headway)
else:
    scores_headway = original_scores

# Intervention 2: add stops
if 'stop_count' in X_sim.columns:
    X_sim_stops = X_sim.copy()
    X_sim_stops['stop_count'] = X_sim_stops['stop_count'] + 5
    scores_stops = xgb_final.predict(X_sim_stops)
else:
    scores_stops = original_scores

# Intervention 3: both combined
X_sim_both = X_sim.copy()
if 'headway_peak_am_min' in X_sim_both.columns:
    X_sim_both['headway_peak_am_min'] = X_sim_both['headway_peak_am_min'] / 2
if 'stop_count' in X_sim_both.columns:
    X_sim_both['stop_count'] = X_sim_both['stop_count'] + 5
scores_both = xgb_final.predict(X_sim_both)

# Display results
sim_results = pd.DataFrame({
    'tract_geoid': df.loc[critical_idx, 'tract_geoid'].values,
    'current_score': original_scores,
    'halve_headway': scores_headway,
    'add_5_stops': scores_stops,
    'both_interventions': scores_both,
    'change_headway_%': ((scores_headway - original_scores) / original_scores * 100),
    'change_stops_%': ((scores_stops - original_scores) / original_scores * 100),
    'change_both_%': ((scores_both - original_scores) / original_scores * 100),
})

print(f"{'='*90}")
print(f"SPRINT 3 SIMULATION DEMO — Top 10 Critical Tracts")
print(f"{'='*90}")
print(f"Interventions: (A) Halve peak AM headway  (B) Add 5 stops  (C) Both A+B")
print()
print(sim_results.to_string(index=False, float_format='{:.4f}'.format))

print(f"\nAverage score change:")
print(f"  Halve headway:     {sim_results['change_headway_%'].mean():+.2f}%")
print(f"  Add 5 stops:       {sim_results['change_stops_%'].mean():+.2f}%")
print(f"  Both interventions:{sim_results['change_both_%'].mean():+.2f}%")
print(f"\nNote: Negative % = improvement (lower equity priority score = better served)")

## 12. Save All Outputs

Export model results, risk assessments, and SHAP values for Sprint 3 and the final report.

In [ ]:
# 1. Save risk assessment
risk_df.to_csv('Sprint2b_Risk_Assessment.csv', index=False)
print(f"Saved: Sprint2b_Risk_Assessment.csv ({len(risk_df)} tracts)")

# 2. Save SHAP importance
shap_importance.to_csv('Sprint2b_SHAP_Importance.csv', index=False)
print(f"Saved: Sprint2b_SHAP_Importance.csv ({len(shap_importance)} features)")

# 3. Save XGBoost model
xgb_final.save_model('Sprint2b_XGBoost_Model.json')
print(f"Saved: Sprint2b_XGBoost_Model.json")

# 4. Save XGBoost CV predictions (for comparison with Ridge)
xgb_pred_df = pd.DataFrame({
    'tract_geoid': df['tract_geoid'].values,
    'equity_tier': tier_labels,
    'true_score': y.values,
    'predicted_score_xgb': y_pred_cv,
    'residual': y.values - y_pred_cv,
})
xgb_pred_df.to_csv('Sprint2b_XGBoost_Predictions.csv', index=False)
print(f"Saved: Sprint2b_XGBoost_Predictions.csv ({len(xgb_pred_df)} tracts)")

# 5. Save fragile tracts specifically
fragile_output = risk_df[risk_df['fragile']].copy() if risk_df['fragile'].any() else risk_df.nlargest(20, 'score_change_pct')
fragile_output.to_csv('Sprint2b_Fragile_Tracts.csv', index=False)
print(f"Saved: Sprint2b_Fragile_Tracts.csv ({len(fragile_output)} tracts)")

print(f"\nAll outputs saved to Sprint 2/ directory.")

## 13. Summary

### Model Performance

| Model | R² (CV) | RMSE (CV) | MAE (CV) | Interpretable? | Simulation-ready? |
|-------|---------|-----------|----------|----------------|-------------------|
| Ridge (Notebook 1) | See Notebook 1 | See Notebook 1 | See Notebook 1 | Yes (coefficients) | No |
| XGBoost (this notebook) | See Section 7 | See Section 7 | See Section 7 | Via SHAP | Yes |

### Risk Scoring

Fragile tracts are identified using a **forecast-then-predict** approach:
1. Project demographic trends (ACS slopes) forward 2-3 years
2. Re-predict equity scores using the trained XGBoost model
3. Flag tracts crossing tier thresholds with >10% score increase

### Sprint 3 Connection

The XGBoost model is saved as `Sprint2b_XGBoost_Model.json` and can be loaded for Sprint 3 scenario simulation. It accepts GTFS features (headway, frequency, span, stops) as inputs, enabling "what-if" analysis of transit service changes.

### Key Files Produced

| File | Description |
|------|-------------|
| `Sprint2b_Risk_Assessment.csv` | All tracts with current + projected scores and fragile flags |
| `Sprint2b_SHAP_Importance.csv` | Feature importance from SHAP |
| `Sprint2b_XGBoost_Model.json` | Saved model for Sprint 3 |
| `Sprint2b_XGBoost_Predictions.csv` | CV predictions for comparison |
| `Sprint2b_Fragile_Tracts.csv` | Fragile tracts detail |